In [ ]:
import pandas as pd 
import yake
from collections import Counter
import nltk
import spacy
from nltk import word_tokenize
from spacy.matcher.matcher import Matcher
import string
import re
import unidecode
import swifter 

class Preprocessing:
    def deEmojify(self,inputString):
        returnString = ""

        for character in inputString:
            try:
                character.encode("ascii")
                returnString += character
            except UnicodeEncodeError:
                try:
                    replaced = unidecode(str(character))
                    if replaced != '':
                        returnString += replaced
                except:
                    pass
                    # print("character:", character)
        return returnString


    def handle_emojis(self,tweet):
        # Smile -- :), : ), :-), (:, ( :, (-:, :')
        tweet = re.sub(r'(:\s?\)|:-\)|\(\s?:|\(-:|:\'\))', '', tweet)
        # Laugh -- :D, : D, :-D, xD, x-D, XD, X-D
        tweet = re.sub(r'(:\s?D|:-D|x-?D|X-?D)', '', tweet)
        # Love -- <3, :*
        tweet = re.sub(r'(<3|:\*)', '', tweet)
        # Wink -- ;-), ;), ;-D, ;D, (;,  (-;
        tweet = re.sub(r'(;-?\)|;-?D|\(-?;)', '', tweet)
        # Sad -- :-(, : (, :(, ):, )-:
        tweet = re.sub(r'(:\s?\(|:-\(|\)\s?:|\)-:)', '', tweet)
        # Cry -- :,(, :'(, :"(
        tweet = re.sub(r'(:,\(|:\'\(|:"\()', '', tweet)
        return tweet


    def preprocess_word(self,word):
        # Remove punctuation
        # word = word.strip('\'"?!,.():;')
        # Convert more than 4 letter repetitions to 2 letter
        # funnnnny --> funny
        word = re.sub(r'(\w)\1{3,}', r'\1\1', word)

        # Remove - & '
        #word = re.sub(r'(-|\')', '', word)

        # Replace numbers with <number>
        # word = re.sub(r'\d+', '<number>', word)
        return word


    def is_valid_word(self,word):
        # return (re.search(r'^[a-z0-9A-Z%][a-z0-9A-Z\._]*$', word) is not None)
        return True

    def preprocess_tweet(self,tweet):
        
        tweet=str(tweet)
        processed_tweet = []
        
        tweet = tweet.lower().strip()

        
        tweet = self.deEmojify(tweet)
        
        tweet = re.sub('’', '\'', tweet)

        
        tweet = re.sub(r'&amp;', 'and', tweet)
        tweet = re.sub(r'&gt;', '>', tweet)
        tweet = re.sub(r'&lt;', '<', tweet)

        
        
        tweet = re.sub(r'((www\.[\S]+)|(https?://[\S]+))', '', tweet) 
        
        tweet = re.sub(r'@([\S]+)', r'', tweet) # re.sub(r'@([\S]+)', r'\1', tweet)
        
        
        tweet = re.sub(r'#(\S+)', r'', tweet) 
        
        tweet = re.sub(r'^\brt\b', '', tweet)
        
        tweet = re.sub(r' rt ', '', tweet)
        
        tweet = re.sub(r'\.{2,}', ' ', tweet)
        
        tweet = tweet.strip(' "\'')
        
        tweet = self.handle_emojis(tweet)
        
        tweet = re.sub(r'\s+', ' ', tweet)
        tweet = re.sub(r'[",():;\-/“]', '', tweet)

        tweet = re.sub(r' \.', '.', tweet)
        tweet = re.sub(r'’', '\'', tweet)

        words = nltk.word_tokenize(tweet)
        for word in words:
            word = self.preprocess_word(word)
            if self.is_valid_word(word):
                processed_tweet.append(word)
        tweet = ' '.join(processed_tweet)

        tweet = tweet.replace("< ", "<")
        tweet = tweet.replace(" >", ">")
        tweet = re.sub(r'\bnt\b', 'not', tweet)
        return tweet

def extract_key_phrases(tweets):
    
    key_phrases_counter = Counter()
    for index, sentence in enumerate(tweets):
        tweet=sentence

        custom_kw_extractor = yake.KeywordExtractor()
        keywords = custom_kw_extractor.extract_keywords(tweet)
        first_elements = [t[0] for t in keywords]
        
        key_phrases_counter.update(first_elements)
        if (index + 1) % 5000 == 0:
            print(f"Key Phrases done {index + 1} rows")
            
            print(f"Number of unique key phrases collected so far: {len(key_phrases_counter)}")
    unique_key_phrases = list(key_phrases_counter.keys())
    key_phrases_counts = list(key_phrases_counter.values())

    return unique_key_phrases, key_phrases_counts

df = pd.read_csv('final_output.csv')
df.rename(columns={'value': 'content'}, inplace=True)

print(df.head())
print(df.shape)
en_df = df#[df['lang'] == 'en']
print(en_df.head())
print(en_df.shape)
unique_content_values = en_df['content'].tolist()
print(len(unique_content_values))
print(unique_content_values[:4])
print("length")
print(len(unique_content_values))
merged_df = pd.DataFrame(unique_content_values, columns=['content'])
preprocess=Preprocessing()
merged_df['processed_tweet_text'] = merged_df['content'].swifter.apply(lambda x: preprocess.preprocess_tweet(x))
merged_df['processed_tweet_text']=merged_df['processed_tweet_text'].str.strip() 
merged_df = merged_df[merged_df['processed_tweet_text'] != '']
unique_content_values = merged_df['processed_tweet_text'].tolist()
a,b=extract_key_phrases(unique_content_values)
df_keywords = pd.DataFrame({'phrase': a, 'tweet_frequency': b})
df_keywords = df_keywords.sort_values(by='tweet_frequency', ascending=False)
df_keywords.to_csv('yake.csv', index=False)

In [ ]:
!pip install yake